# CV Masterclass 10: Multimodal Vision

Welcome to Notebook 13. Computer Vision does not exist in a vacuum. A model that can draw a bounding box around a cat is useful, but a model that can *explain* what the cat is doing and segment exactly the objects you describe in free-text is transformative.

Multimodal Vision bridges the gap between Pixels and Text.

---

## 🎓 Socratic Deep Check
Ponder this question before we begin:

> *"Grounding DINO merges a vision backbone (Swin) and a language backbone (BERT) via deep cross-attention layers at multiple scales in the network. SAM, however, uses a strict prompt encoder where text/points are injected ONLY at the end into a lightweight mask decoder, strictly AFTER the massive ViT-H finishes its entire spatial forward pass. Why does SAM use this 'late fusion' geometry, and how does it explicitly enable 50-millisecond interactive real-time prompting in a web browser, whereas Grounding DINO cannot?"*

> *"In LLaVA-NeXT (AnyRes), a high-resolution image is sliced into sub-patches, each processed by a ViT. If we have a 672x672 image and a ViT with 336x336 input, we get a 2x2 grid. We also process a downsampled 336x336 version of the whole image. Why is this 'global context' patch mathematically essential for the LLM's spatial reasoning, and how does it prevent the model from 'losing the forest for the trees'?"
*

> *"Set-of-Mark (SoM) Prompting overlays numbered bounding boxes on every interactive element before passing the image to an LLM. While this drastically improves grounding accuracy, it adds a pre-processing latency (running a segmenter like SAM). If you had to build a real-time 'low-latency' visual agent that must respond in <100ms, would you still use SoM, or would you favor 'Action Regression' where the LLM predicts raw screen coordinates directly, and what are the primary architectural risks of the latter approach?"*
---

1. **CLIP:** Contrastive Language-Image Pretraining.
2. **BLIP-2 & LLaVA:** Instruction Tuning & The Q-Former.
3. **LLaVA-NeXT (AnyRes):** Dynamic High-Resolution Grids.
4. **SAM (Segment Anything Model):** Promptable Masks operations.
5. **Grounding DINO:** Open-Set Object Detection.
6. **Vision-Language-Action (VLA):** Robotics & Action Tokenization.
7. **Agentic GUI Vision:** Visual Web Agents and Set-of-Mark (SoM) Prompting.


## 1. CLIP Architecture and Training

In 2021, OpenAI released CLIP, which forever altered computer vision. It is technically an embedding alignment system.

**The Architecture:**
1.  **Image Encoder:** A Vision Transformer (e.g., ViT-L/14) mapping an image to a $D=512$ dimensional vector.
2.  **Text Encoder:** A GPT-style Transformer mapping a sentence (e.g., *"A photo of a dog playing"*) to a $D=512$ dimensional vector.

**The Math (InfoNCE Loss):**
Given a batch of $N$ image-text pairs, CLIP computes the Cosine Similarity between *every* image and *every* text snippet, creating an $N \times N$ similarity matrix.

The **Symmetric Contrastive Loss** maximizes the values on the diagonal (true pairs) and minimizes the off-diagonal values (false pairs).
$$ L_{image} = -\log \frac{\exp(I_i \cdot T_i / \tau)}{\sum_{j=1}^N \exp(I_i \cdot T_j / \tau)} $$
$$ L_{text} = -\log \frac{\exp(I_i \cdot T_i / \tau)}{\sum_{j=1}^N \exp(I_j \cdot T_i / \tau)} $$
$$ L = \frac{1}{2}(L_{image} + L_{text}) $$

Because the model trains on 400 Million pairs scraped from the internet, it achieves breathtaking zero-shot accuracy—meaning it can classify an image without ever having seen an explicit "label" for it!


## 📑 Table of Contents

* [🎓 Socratic Deep Check](#socratic-deep-check)
* [1. CLIP Architecture and Training](#1-clip-architecture-and-training)
  * [1.1 InfoNCE Loss for CLIP Training](#11-infonce-loss-for-clip-training)
  * [COMMON PITFALLS](#common-pitfalls)
* [2. BLIP-2 and Instruction Tuning](#2-blip-2-and-instruction-tuning)
  * [COMMON PITFALLS](#common-pitfalls)
* [LLaVA (Large Language and Vision Assistant)](#llava-large-language-and-vision-assistant)
* [3. LLaVA-NeXT and AnyRes (High-Resolution)](#3-llava-next-and-anyres-high-resolution)
  * [COMMON PITFALLS](#common-pitfalls)
* [4. SAM (Segment Anything Model)](#4-sam-segment-anything-model)
  * [🎓 DEEP QUESTION ANSWERED](#deep-question-answered)
  * [COMMON PITFALLS](#common-pitfalls)
* [5. Grounding DINO](#5-grounding-dino)
  * [Why Grounding DINO Cannot Use Late Fusion](#why-grounding-dino-cannot-use-late-fusion)
  * [COMMON PITFALLS](#common-pitfalls)
* [6. Vision-Language-Action (VLA)](#6-vision-language-action-vla)
  * [Action Tokenization](#action-tokenization)
  * [COMMON PITFALLS](#common-pitfalls)
* [7. Agentic GUI Vision & Visual Web Agents](#7-agentic-gui-vision-visual-web-agents)
  * [The Challenge of UI Parsing](#the-challenge-of-ui-parsing)
  * [Set-of-Mark (SoM) Prompting](#set-of-mark-som-prompting)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# -----------------------------------------------------
# IMPLEMENTATION: Zero-Shot Classification & Retrieval
# -----------------------------------------------------
class DummyCLIP(nn.Module):
    def __init__(self, patch_size=16, img_size=224, d_out=512, txt_vocab=128):
        super().__init__()
        n_patches = (img_size // patch_size) ** 2  # = 196
        patch_dim = 3 * patch_size * patch_size    # = 768
        # Image encoder: patches -> embedding (simplified ViT)
        self.patch_embed = nn.Linear(patch_dim, d_out)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_out))
        self.img_proj = nn.Linear(d_out, d_out)
        # Text encoder: token sequence -> embedding (simplified)
        self.txt_embed = nn.Embedding(txt_vocab, d_out)
        self.txt_proj = nn.Linear(d_out, d_out)
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1/0.07))
    
    def encode_image(self, x):
        # x: [B, 3, H, W] → patchify → [B, N, patch_dim]
        B, C, H, W = x.shape
        p = 16
        x = x.reshape(B, C, H//p, p, W//p, p).permute(0,2,4,1,3,5)
        x = x.reshape(B, -1, C*p*p)  # [B, 196, 768]
        x = self.patch_embed(x).mean(dim=1)  # Average pool patches
        return F.normalize(self.img_proj(x), dim=-1)
    
    def encode_text(self, t):
        # t: [B, seq_len] integer token indices
        x = self.txt_embed(t).mean(dim=1)
        return F.normalize(self.txt_proj(x), dim=-1)
    
    def forward(self, images, text_tokens):
        img_feat = self.encode_image(images)
        txt_feat = self.encode_text(text_tokens)
        scale = self.logit_scale.exp()
        return scale * img_feat @ txt_feat.T, scale * txt_feat @ img_feat.T

# TEST IT
model = DummyCLIP()
images = torch.randn(3, 3, 224, 224)
text_tokens = torch.randint(0, 128, (5, 20))  # 5 captions, 20 tokens each
logits_i2t, logits_t2i = model(images, text_tokens)
assert logits_i2t.shape == (3, 5), f"Expected (3,5), got {logits_i2t.shape}"
assert logits_t2i.shape == (5, 3), f"Expected (5,3), got {logits_t2i.shape}"

# 1. Zero-Shot Classification: Given an image, which text describes it best?
probs = logits_i2t.softmax(dim=-1) # shape: [3 images, 5 potential labels]
predicted_labels = probs.argmax(dim=-1)
print(f"Zero-Shot Classification - Given 3 images, the predicted label IDs are: {predicted_labels.tolist()}")

# 2. Zero-Shot Retrieval: Given Text Query #2, which image is it?
text_query_idx = 2
image_retrieval_probs = logits_i2t[:, text_query_idx].softmax(dim=0) # [3 potential images]
best_image_match = image_retrieval_probs.argmax().item()
print(f"Zero-Shot Retrieval - Given text query '{text_query_idx}', Image #{best_image_match} is the closest match.")

### 1.1 InfoNCE Loss for CLIP Training

The symmetric InfoNCE loss ensures that the similarity of matching pairs is maximized while minimizing the similarity of all other combinations in the batch.

In [ ]:
def clip_contrastive_loss(logits_i2t, logits_t2i):
    """Symmetric InfoNCE loss for CLIP training."""
    N = logits_i2t.shape[0]
    labels = torch.arange(N, device=logits_i2t.device)
    loss_i2t = F.cross_entropy(logits_i2t, labels)
    loss_t2i = F.cross_entropy(logits_t2i, labels)
    return (loss_i2t + loss_t2i) / 2

# TEST IT
batch_size = 32
logits_i2t = torch.randn(batch_size, batch_size)
logits_t2i = logits_i2t.T
loss = clip_contrastive_loss(logits_i2t, logits_t2i)
print(f"Contrastive Loss: {loss.item():.4f}")
assert loss > 0

# Verify loss decreases if matching pairs are more similar
logits_i2t_similar = logits_i2t.clone()
diag_indices = torch.arange(batch_size)
logits_i2t_similar[diag_indices, diag_indices] += 10.0
loss_similar = clip_contrastive_loss(logits_i2t_similar, logits_i2t_similar.T)
print(f"Loss with similar matching pairs: {loss_similar.item():.4f}")
assert loss_similar < loss

### COMMON PITFALLS
*   **The Zero-Shot Class Embedding Bottleneck:** When doing zero-shot classification, users often just feed the bare word `"dog"`. Accuracy increases radically if you use prompt engineering to template the text embedding: `"a photo of a {dog}, an animal."` This aligns the text string closer to the natural syntax distribution the model saw during pre-training.


## 2. BLIP-2 and Instruction Tuning

If an Image is passed into a Large Language Model (like LLaMA or T5), the LLM expects discrete text tokens, not a massive grid of continuous pixel embeddings. How do we bridge them?

**BLIP-2 Solution (The Q-Former):**
BLIP-2 introduces the Querying Transformer (Q-Former). It is a lightweight bridge consisting of exactly 32 fixed query vectors. 
These 32 queries use Cross-Attention to interact with the output of a completely *frozen* ViT, extracting solely the visual information that is most critical.

This architecture enables bridging state-of-the-art frozen vision encoders with state-of-the-art frozen LLMs (like OPT or FlanT5) simultaneously without fine-tuning either of the massive models!

**The Two-Stage Bootstrap Training:**
1.  **Stage 1:** Bootstrap Vision-Language Representation. The Q-Former is trained connected to the frozen ViT using contrastive loss, image-grounded text generation, and image-text matching.
2.  **Stage 2:** Bootstrap LLM Interaction. The Q-Former is connected to the frozen LLM. The 32 extracted visual query tokens are linearly projected directly into the text embedding space of the LLM and prepended as "soft prompt" instructions before the actual text input.

Because only the tiny Q-Former is updated, it trains with stunning computational efficiency.


In [ ]:
import torch.nn as nn

# -----------------------------------------------------
# IMPLEMENTATION: The Q-Former Conceptual Bridge
# -----------------------------------------------------

class MinimalQFormer(nn.Module):
    def __init__(self, vit_dim=1024, text_embed_dim=768, num_queries=32):
        super().__init__()
        # 32 trainable query tokens that will "look" at the image
        self.query_tokens = nn.Parameter(torch.randn(1, num_queries, text_embed_dim))
        
        # Cross-Attention to look at the frozen ViT
        self.cross_attention = nn.MultiheadAttention(embed_dim=text_embed_dim, kdim=vit_dim, vdim=vit_dim, num_heads=8, batch_first=True)
        
        # Projection to the LLM's expected dimensional space
        self.llm_proj = nn.Linear(text_embed_dim, text_embed_dim)
        
    def forward(self, frozen_vit_features):
        B = frozen_vit_features.size(0)
        # Expand queries to batch size
        queries = self.query_tokens.expand(B, -1, -1)
        
        # The queries extract visual information conditioned on the image
        extracted_visuals, _ = self.cross_attention(query=queries, key=frozen_vit_features, value=frozen_vit_features)
        
        # Project into the LLM textual space
        soft_prompt = self.llm_proj(extracted_visuals)
        return soft_prompt

# TEST IT
B = 2
vit_features = torch.randn(B, 196, 1024) # Frozen ViT outputs 196 patches
qformer = MinimalQFormer()
llm_input_embeddings = qformer(vit_features)

print(f"Q-Former Token Extractor shape: {llm_input_embeddings.shape}") 
print("These 32 tokens will be prepended to the frozen LLM text sequence as visual instructions!")


### COMMON PITFALLS
*   **Catastrophic Forgetting during End-to-End Tuning:** If you unfreeze the LLM or ViT during Q-Former training, the model immediately forgets standard English syntax or spatial features. Keeping them frozen enforces strict structural boundaries.


## LLaVA (Large Language and Vision Assistant)

LLaVA is the most widely deployed open-source MLLM because of its dramatic architectural simplicity compared to BLIP-2.

**Key Architectural features:**
- Architecture is dramatically simpler than BLIP-2.
- A frozen CLIP ViT-L/14 extracts image features.
- A single linear projection layer (or 2-layer MLP in LLaVA 1.5) maps CLIP features to the LLM's embedding dimension.
- A Vicuna or LLaMA LLM takes the projected visual tokens + text tokens concatenated as input.
- **NO Q-Former.** NO frozen LLM. The entire LLM is instruction-tuned.

**Comparison Table:**

| Model   | Vision Encoder | Bridge         | LLM     | LLM Frozen? |
|---------|----------------|----------------|---------|-------------|
| BLIP-2  | ViT-G (frozen) | Q-Former       | OPT/T5  | YES         |
| LLaVA   | CLIP ViT-L     | Linear/MLP     | Vicuna  | NO          |

In [ ]:
class LLaVAProjection(nn.Module):
    """
    LLaVA 1.5 uses a 2-layer MLP instead of LLaVA 1.0's linear layer.
    Maps CLIP visual features into LLaMA's embedding space.
    """
    def __init__(self, clip_dim=1024, llm_dim=4096):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(clip_dim, llm_dim),
            nn.GELU(),
            nn.Linear(llm_dim, llm_dim)
        )
    def forward(self, clip_features):
        return self.proj(clip_features)

# TEST IT
clip_feats = torch.randn(1, 256, 1024)  # 256 CLIP visual tokens
bridge = LLaVAProjection()
llm_visual_tokens = bridge(clip_feats)
print(f"Visual tokens ready for LLaMA: {llm_visual_tokens.shape}")
# These 256 tokens are prepended to text tokens in the LLM

## 3. LLaVA-NeXT and AnyRes (High-Resolution)

Standard Vision Transformers (ViTs) are often locked to a fixed input resolution (e.g., $224\times 224$ or $336\times 336$). When you feed a 4K document into these models, they resize it, obliterating small text and fine details.

**LLaVA-NeXT (AnyRes)** solves this with a dynamic path-based strategy.

**The Mechanism:**
1.  **Image Slicing:** A high-res image is divided into sub-patches (e.g., $2\times 2$ grid) that match the ViT's native resolution.
2.  **Global Path:** A downsampled version of the entire image is also processed to provide global spatial context.
3.  **Token Pooling:** Each patch generates $14\times 14 = 196$ (or $24\times 24 = 576$) tokens. These are concatenated along with a special 'newline' token to help the LLM maintain the 2D spatial arrangement.

**The Math:**
If the base resolution is $N\times N$, an AnyRes model can handle $M \cdot N \times K \cdot N$ by processing $M \times K$ independent patches. The total token count $T$ becomes:
$$ T = (M \times K + 1) \times P + (M - 1) \times L $$
Where $P$ is patches per sub-image and $L$ is the number of 'newline' delimiter tokens. This allows for essentially 'infinite' resolution theoretically, bounded only by the LLM's context window.


In [ ]:
import torch
import torch.nn as nn

# -----------------------------------------------------
# IMPLEMENTATION: AnyRes Grid Slicing Logic
# -----------------------------------------------------

class AnyResEncoder(nn.Module):
    def __init__(self, vit_backbone, base_res=336):
        super().__init__()
        self.vit = vit_backbone # Mock ViT
        self.base_res = base_res

    def slice_image(self, x, grid=(2, 2)):
        # x: [B, C, 672, 672] -> Split into 4 patches of 336x336
        B, C, H, W = x.shape
        gh, gw = grid
        ph, pw = H // gh, W // gw
        
        patches = x.unfold(2, ph, ph).unfold(3, pw, pw)
        # patches: [B, C, gh, gw, ph, pw]
        patches = patches.permute(0, 2, 3, 1, 4, 5).contiguous()
        return patches.view(B * gh * gw, C, ph, pw)

    def forward(self, high_res_img):
        # 1. Global Context (Downsample to base_res)
        global_context = torch.nn.functional.interpolate(
            high_res_img, size=(self.base_res, self.base_res), mode='bilinear')
        
        # 2. Local Patches
        local_patches = self.slice_image(high_res_img, grid=(2, 2))
        
        # 3. Features
        # In real LLaVA-NeXT, we run ViT on all 5 images (4 patches + 1 global)
        global_feats = torch.randn(1, 576, 1024) # Placeholder for ViT(global)
        patch_feats = torch.randn(4, 576, 1024)  # Placeholder for ViT(patches)
        
        # 4. Assembly (simplified)
        combined = torch.cat([global_feats, patch_feats.view(1, -1, 1024)], dim=1)
        return combined

# TEST IT
mock_vit = nn.Identity()
encoder = AnyResEncoder(mock_vit)
image_672 = torch.randn(1, 3, 672, 672)

tokens = encoder(image_672)
print(f"High-Res Token sequence length: {tokens.shape[1]}")
print("Tokens consist of: 576 (Global) + 4 * 576 (Patches) = 2880 tokens")
assert tokens.shape[1] == 576 + (4 * 576)


### COMMON PITFALLS
*   **The Aspect Ratio Distortion:** If you slice a 16:9 image into a 2x2 grid, you induce massive stretching or require padding. LLaVA-NeXT uses dynamic grids (e.g., 1x3 or 2x2) to match the nearest aspect ratio of the original image, minimizing spatial deformation.


## 4. SAM (Segment Anything Model)

**Promptable Segmentation:** SAM is designed to cut out anything using three interactive modes: Point clicks, Bounding boxes, or Text.

**The Architecture:**
1.  **Image Encoder:** A colossal MAE-pretrained ViT-H. It processes the image once and outputs a dense spatial embedding.
2.  **Prompt Encoder:** Encodes spatial points/boxes into positional embeddings, and text into CLIP embeddings.
3.  **Mask Decoder:** A tiny, extremely fast transformer block that takes the ViT embeddings and Prompt embeddings, resolving them into physical masks.

### 🎓 DEEP QUESTION ANSWERED
**Q:** *Why does SAM use late fusion, and how does it explicitly enable 50ms interactive web prompting?*

**A:** 
If the text/points were injected early (like Grounding DINO), the massive ViT-H ($>600\text{M}$ parameters) would have to recalculate its entire feature hierarchy every time the user clicked a new point.
By rendering the heavy ViT only *once* (taking $\sim 1$ second) and storing the output embedding, the browser only needs to run the tiny Mask Decoder ($Math\sim 4\text{M}$ params) locally. When you click, the Mask Decoder merges your click with the cached ViT features in $\sim 50\text{ms}$, creating true real-time interactivity.

**The SA-1B Dataset Expansion Strategy:**
SAM is state-of-the-art because of its data engine:
1.  Human annotators used an early SAM to click and generate masks interactively.
2.  The model retrained on those masks.
3.  Eventually, SAM generated masks completely automatically, yielding **SA-1B: 1.1 billion high-quality masks from 11 million images**.

**Open-Vocabulary SAM:**
SAM provides the structural Mask. CLIP provides the semantic label.
If you combine them: SAM generates 100 blind masks for a room. You individually crop each mask and pass it through CLIP to classify it. You instantly have an Open-Vocabulary segmented room!


In [ ]:
# -----------------------------------------------------
# IMPLEMENTATION: SAM Concept Wrappers
# -----------------------------------------------------
# Simulating the official Meta SDK execution pipeline

class SAM_SDK_Simulator:
    def set_image(self, rgb_image):
        print(f"Running massive ViT on image shape {rgb_image.shape}. This takes 1.0 seconds.")
        # Simulates returning the cached image embedding spatial matrix
        self.cached_embedding = torch.randn(1, 256, 64, 64) 
        
    def predict(self, point_coords, point_labels):
        print(f"Injecting point {point_coords} into tiny Mask Decoder!")
        # Simulates 50ms real time decoding
        generated_mask = (torch.randn(256, 256) > 0.0).float()
        score = 0.95
        return generated_mask, score
        
    def generate_all(self):
        print("Automatic Mask Generation: Casting a 32x32 grid of points across the image to find all objects...")
        return {"masks": [torch.ones(10, 10)] * 142} # Returns list of cropped mask dicts

# TEST IT
sam_predictor = SAM_SDK_Simulator()

import numpy as np
dummy_frame = np.zeros((1024, 1024, 3))

# 1. Initialize image once
sam_predictor.set_image(dummy_frame)

# 2. Real time interactivity
mask, conf = sam_predictor.predict(point_coords=[[500, 300]], point_labels=[1])
print(f"Interactive Mask Generated with {conf*100}% confidence.")

# 3. Grid Generation
all_entities = sam_predictor.generate_all()
print(f"Auto-generated {len(all_entities['masks'])} object masks across the scene.")


### COMMON PITFALLS
*   **The Ambiguity Failure:** If you click on the steering wheel of a car, does SAM segment the steering wheel, or the whole car? It mathematically predicts 3 layered masks (Sub-part, Object, Semantic Group) to solve ambiguity, and you must script logic to choose the right depth for your task.


## 5. Grounding DINO

Standard Object Detection (like YOLOv8) is a **Closed-Set** problem. It is trained on the COCO dataset, which possesses exactly 80 predefined classes (Dog, Cat, Car). If you show it a "Lightsaber", it fundamentally lacks the output nodes to identify it.

**Grounding DINO** solves **Open-Set Object Detection**. 
Instead of a fixed class vocabulary, the detector takes an Image and a Free-Text string (e.g. *"A blue lightsaber, a cup of coffee on the desk"*). It outputs bounding boxes targeting exactly those text mentions.

**The Architecture Engine:**
It fuses a Swin Transformer (Vision) with a BERT (Text) encoder using intensive Cross-Attention layers at every scale. 
*The Impact:* A YOLO knows 80 classes. Grounding DINO understands every noun and descriptive adjective phrase in the English language inherently because of the BERT contrastive embeddings. 


### Why Grounding DINO Cannot Use Late Fusion

- **CLIP and SAM use Late Fusion:** encode image → encode text separately → compare at the end via dot product.
- Late fusion cannot detect "the LEFT cat" vs "the RIGHT cat" — the spatial relationship between language tokens and image regions is lost.
- **Grounding DINO uses Deep Fusion:** at every FPN scale (P3, P4, P5), vision features and language features cross-attend to each other. The text token "left" modulates the spatial attention weights of the vision backbone at the corresponding spatial location.
- This is why Grounding DINO can localize phrases (not just classify images) but is 10-20x slower than CLIP's retrieval.

In [ ]:
class BiDirectionalCrossAttention(nn.Module):
    """Simplified version of the Grounding DINO feature enhancement."""
    def __init__(self, v_dim, l_dim, embed_dim):
        super().__init__()
        # Vision attends to language
        self.v_to_l_attn = nn.MultiheadAttention(v_dim, num_heads=8,
                                                  kdim=l_dim, vdim=l_dim,
                                                  batch_first=True)
        # Language attends to vision
        self.l_to_v_attn = nn.MultiheadAttention(l_dim, num_heads=8,
                                                  kdim=v_dim, vdim=v_dim,
                                                  batch_first=True)
    def forward(self, v_feat, l_feat):
        # Vision features enhanced by language context
        v_out, _ = self.v_to_l_attn(v_feat, l_feat, l_feat)
        # Language features enhanced by vision context
        l_out, _ = self.l_to_v_attn(l_feat, v_feat, v_feat)
        return v_feat + v_out, l_feat + l_out

# TEST IT
v_feat = torch.randn(1, 196, 256)  # Visual tokens from FPN
l_feat = torch.randn(1, 20, 256)   # Language tokens from BERT
bi_cross = BiDirectionalCrossAttention(256, 256, 256)
v_enhanced, l_enhanced = bi_cross(v_feat, l_feat)
assert v_enhanced.shape == v_feat.shape
assert l_enhanced.shape == l_feat.shape
print(f"Fused Vision features: {v_enhanced.shape}")
print(f"Fused Language features: {l_enhanced.shape}")

In [ ]:
# -----------------------------------------------------
# IMPLEMENTATION: Grounding DINO Pipeline Logic
# -----------------------------------------------------
import torch

class OpenSetDetectorSimulator:
    def __init__(self):
        print("Loading Swin+BERT Cross-Attention Engine...")
        
    def predict(self, image, text_prompt, box_threshold, text_threshold):
        # The prompt is tokenized
        tokens = text_prompt.split(", ")
        
        print(f"Scanning image for text tokens: {tokens}")
        print(f"Applying spatial Box Threshold: {box_threshold}")
        print(f"Applying semantic Text Threshold: {text_threshold}")
        
        # Simulates bounding box outputs logic [x1, y1, x2, y2]
        boxes = torch.tensor([[100, 150, 400, 500]])
        labels = ["blue lightsaber"]
        scores = torch.tensor([0.92])
        return boxes, labels, scores

# TEST IT
gdino = OpenSetDetectorSimulator()
img_t = torch.randn(3, 512, 512)
text = "a blue lightsaber, a cup of coffee"

boxes, labels, confs = gdino.predict(img_t, text, box_threshold=0.35, text_threshold=0.25)
print(f"Detected: {labels[0]} at Box Bounds {boxes.tolist()[0]} confident {confs[0].item():.2f}")
# If you combined Grounding DINO's Box with SAM's interactive box-prompt... you achieve Zero-Shot perfectly cut semantic masks!


### COMMON PITFALLS
*   **Assuming Speed Parity with YOLO:** Grounding DINO has to run intensive BERT multi-scale cross attention dynamically per-image because the text input dictates the actual spatial hierarchy. It generally runs an order of magnitude slower than native YOLO models unless heavily compiled.


## 6. Vision-Language-Action (VLA)

How do we take a model that "sees and talks" and make it "do"? **VLA models** (like RT-2 or OpenVLA) treat robotic control as a translation problem.

### Action Tokenization
Robotic arms are typically controlled by 7-DOF (Degree of Freedom) vectors: $(x, y, z, \text{pitch}, \text{yaw}, \text{roll}, \text{gripper})$. Instead of training a separate regression head (which is brittle), VLA models **discretize** these values into text tokens.

**The Logic:**
1.  Scale each dimension to an integer range (e.g., 0-255).
2.  Map each integer to an unused or specific token in the LLM's vocabulary.
3.  The LLM outputs action tokens just like it outputs words.

**Why this works?**
By using the LLM's internal weights, the model can leverage semantic reasoning. For example, if you ask it to "Pick up the apple", the model identifies the apple (Visual), reasons that it needs to move its gripper (Language/Logic), and then outputs the exact tokens to move there (Action).


In [ ]:
import numpy as np

# -----------------------------------------------------
# IMPLEMENTATION: Action Tokenizer (RT-2 Style)
# -----------------------------------------------------

class ActionTokenizer:
    def __init__(self, bins=256):
        self.bins = bins
        
    def encode(self, actions):
        # actions: [X, Y, Z, P, Y, R, G] in range [-1, 1]
        # Scaled to [0, bins-1]
        discretized = ((actions + 1) / 2 * (self.bins - 1)).astype(int)
        tokens = [f"<action_{v}>" for v in discretized]
        return " ".join(tokens)
    
    def decode(self, token_str):
        # "<action_128> <action_64> ..."
        parts = token_str.replace("<action_", "").replace(">", "").split()
        discretized = np.array([int(p) for p in parts])
        actions = (discretized / (self.bins - 1)) * 2 - 1
        return actions

# TEST IT
tokenizer = ActionTokenizer()
raw_action = np.array([0.5, -0.2, 0.1, 0.0, 0.8, -0.9, 1.0]) # Example SE(3) + Gripper

token_string = tokenizer.encode(raw_action)
print(f"Robotic Action as Text Tokens: {token_string}")

recovered_action = tokenizer.decode(token_string)
print(f"Recovered continuous values: {recovered_action}")

np.testing.assert_almost_equal(raw_action, recovered_action, decimal=2)


### COMMON PITFALLS
*   **The Precision Gap:** Discretizing actions into 256 bins might be too coarse for precise tasks like threading a needle. Modern VLAs use higher bin counts or 'residual' action heads to regain sub-millimeter precision.


## 7. Agentic GUI Vision & Visual Web Agents

How do we take a model that can describe an image and turn it into a model that can **browse the web** or **operate a computer**? This requires shifting from global image descriptors to **fine-grained spatial grounding** on User Interfaces (UIs).

### The Challenge of UI Parsing
Standard MLLMs (like GPT-4V or LLaVA) often struggle with exact coordinate precision. If an LLM is asked to "Click the Login button," it might hallucinate coordinates that are slightly off, causing the mouse to click empty space.

### Set-of-Mark (SoM) Prompting
**Set-of-Mark (SoM)** is a prompting strategy that simplifies the visual grounding task for the LLM by offloading the spatial localization to an expert segmenter (like SAM or a specialized UI detector).

**The Workflow:**
1.  **Preprocessing:** An interactive-element detector scans the screenshot and identifies all buttons, text fields, and links.
2.  **Overlay:** The system overlays a distinctive **colored mask** and a **numbered ID (e.g., [1], [2], [3])** over every detected element.
3.  **Prompting:** The LLM receives the "marked" image and a prompt like: *"You are a web agent. To log in, which element should you click?"*
4.  **Reasoning:** Instead of predicting raw $(x, y)$ coordinates, the LLM simply outputs: *"I will click on element [5]."*

This approach bridges the gap between high-level reasoning and pixel-perfect execution, making autonomous visual agents significantly more robust.

In [ ]:
import torch
import numpy as np
import cv2

# -----------------------------------------------------
# IMPLEMENTATION: Set-of-Mark (SoM) Simulator
# -----------------------------------------------------

class SoMOverlayGenerator:
    """
    Simulates an SoM pre-processor that identifies UI elements
    and overlays numbered marks for LLM grounding.
    """
    def __init__(self, width=1024, height=768):
        self.w = width
        self.h = height
        
    def generate_ui_marks(self, screenshot, detections):
        # detections: List of [x1, y1, x2, y2] boxes
        overlay = screenshot.copy()
        
        for i, box in enumerate(detections):
            x1, y1, x2, y2 = box
            # 1. Draw a semi-transparent mask
            color = (np.random.randint(0, 255), np.random.randint(0, 255), np.random.randint(0, 255))
            cv2.rectangle(overlay, (x1, y1), (x2, y2), color, -1)
            
            # 2. Overlay a numbered ID
            label = f"[{i}]"
            cv2.putText(overlay, label, (x1, y1 - 5), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            
        # Blend original and mask
        alpha = 0.3
        marked_screenshot = cv2.addWeighted(overlay, alpha, screenshot, 1 - alpha, 0)
        return marked_screenshot

# TEST IT
screenshot = np.ones((768, 1024, 3), dtype=np.uint8) * 200 # Grey UI
generator = SoMOverlayGenerator()

# Mock detections: [Login Button, Search Bar, Submit]
ui_elements = [
    [100, 100, 250, 150], 
    [300, 100, 700, 150], 
    [750, 100, 900, 150]
]

som_image = generator.generate_ui_marks(screenshot, ui_elements)
print(f"SoM Processed Image shape: {som_image.shape}")
print("LLM Input: 'The Search Bar is element [1]. Type 'VideoMAE' into it.'")
assert som_image.shape == (768, 1024, 3)
